In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import random
import time
import json
import os

from IPython.display import display
import matplotlib
matplotlib.get_backend()

'module://matplotlib_inline.backend_inline'

In [ ]:
%matplotlib widget

In [ ]:
import torch
import torch.nn as nn

class LinearIntegrator(nn.Module):
    def __init__(self, k=5):
        super().__init__()
        self.lin = nn.Linear(k * 3, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)


class MLPIntegrator(nn.Module):
    def __init__(self, k=5, hidden=128, num_layers=3):
        super().__init__()
        in_dim = k * 3

        layers = [nn.Linear(in_dim, hidden), nn.ReLU()]

        for _ in range(num_layers - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU()]

        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def load_integrator_model(path, device="cuda"):
    """
    Load a trained linear or MLP integrator saved in the Square format.
    """
    device = torch.device(device)

    ckpt = torch.load(path, map_location=device)

    model_type = ckpt["model_type"]
    cfg = ckpt["config"]

    if model_type == "linear":
        model = LinearIntegrator(k=cfg["k"])

    elif model_type == "mlp":
        model = MLPIntegrator(
            k=cfg["k"],
            hidden=cfg["hidden"],
            num_layers=cfg["num_layers"],
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device).eval()

    print(f"Loaded {model_type} integrator")
    print(f"  k       = {cfg['k']}")
    print(f"  R       = {cfg['R']}")
    print(f"  u_scale = {cfg['u_scale']:.6e}")

    return model, cfg

In [ ]:
def build_bc_mask(Nt, Nx):
    is_bc = np.zeros((Nt, Nx), dtype=bool)
    is_bc[:, 0] = True
    is_bc[:, Nx-1] = True
    return is_bc

def activate_bc_from_squares(visited, is_bc, R):
    """
    Activate BC points only if they lie inside a square
    of radius R around any visited *interior* point.
    """
    Nt, Nx = visited.shape
    interior = np.argwhere(visited & (~is_bc))

    for (t0, x0) in interior:
        t_min = max(0, t0 - R)
        t_max = min(Nt - 1, t0 + R)
        x_min = max(0, x0 - R)
        x_max = min(Nx - 1, x0 + R)

        for t in range(t_min, t_max + 1):
            for x in range(x_min, x_max + 1):
                if is_bc[t, x]:
                    visited[t, x] = True

In [ ]:
def build_features_for_model(neigh, sampled, u_values, u_scale):
    """
    Build Scheme-B features from already-selected neighbors.

    neigh: list or array of shape (k, 2) with (tn, xn)
    sampled: (t0, x0)
    u_values: dict {(t, x): u}
    u_scale: float (same as training)

    returns:
        feats: np.ndarray shape (k*3,)
    """
    t0, x0 = sampled
    feats = []

    for (tn, xn) in neigh:
        dt = t0 - tn          # NO scaling
        dx = x0 - xn          # NO scaling
        u  = u_values[(int(tn), int(xn))] / u_scale
        feats.append([dt, dx, u])

    return np.array(feats, dtype=np.float32).reshape(-1)

In [ ]:
try:
    import mplcursors

    def visualize_rollout_interactive_simple(
        visited,
        u_values,
        history,
        ground_truth_data,
        config=None,
    ):
        """
        Simple interactive visualization using mplcursors.
        Hover shows error, prediction, neighbors, etc.
        """
        Nt, Nx = visited.shape

        fig, ax = plt.subplots(figsize=(12, 8))

        visited_pts = np.array(list(u_values.keys()))
        history_dict = {tuple(h['sampled']): h for h in history}

        ic_bc_pts = []
        pred_pts = []
        pred_vals = []

        for pt in visited_pts:
            pt = tuple(pt)
            if pt in history_dict:
                pred_pts.append(pt)
                pred_vals.append(u_values[pt])
            else:
                ic_bc_pts.append(pt)

        # ---- IC / BC ----
        if ic_bc_pts:
            arr = np.array(ic_bc_pts)
            ax.scatter(
                arr[:,1], arr[:,0],
                s=20, c="gray", alpha=0.5,
                label="IC / BC", zorder=1
            )

        # ---- Predicted interior ----
        if pred_pts:
            pts = np.array(pred_pts)
            vals = np.array(pred_vals)

            scatter = ax.scatter(
                pts[:,1], pts[:,0],
                c=vals, s=40,
                cmap="viridis",
                edgecolors="black",
                linewidth=0.5,
                alpha=0.8,
                label="Predicted",
                zorder=2,
            )
            plt.colorbar(scatter, ax=ax, label="Predicted u")

            cursor = mplcursors.cursor(scatter, hover=True)

            @cursor.connect("add")
            def on_add(sel):
                idx = sel.index
                pt = tuple(pts[idx])
                h = history_dict.get(pt)
                if h:
                    pred = h["prediction"]
                    true = h["ground_truth"]
                    err = pred - true
                    neigh = h["neighbors"]
                    sel.annotation.set_text(
                        f"Point: {pt}\n"
                        f"Pred: {pred:.4f}\n"
                        f"True: {true:.4f}\n"
                        f"Error: {err:.4f}\n"
                        f"Neighbors: {len(neigh)}"
                    )

        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title("Rollout predictions (hover for details)")
        ax.legend()
        plt.tight_layout()
        plt.show()

    print("mplcursors available — hover visualization enabled")

except ImportError:
    print("mplcursors not available. Install with: pip install mplcursors")

In [ ]:
def visualize_rollout_simple(visited, u_values, history, ground_truth_data, R, config=None):
    """
    Simple non-interactive visualization of rollout results.
    """
    Nt, Nx = visited.shape

    fig, ax = plt.subplots(figsize=(12, 8))

    visited_pts = np.array(list(u_values.keys()))
    history_dict = {tuple(h['sampled']): h for h in history}

    ic_bc_pts = []
    pred_pts = []
    pred_vals = []

    for pt in visited_pts:
        pt_t = tuple(pt)
        if pt_t in history_dict:
            pred_pts.append(pt_t)
            pred_vals.append(u_values[pt_t])
        else:
            ic_bc_pts.append(pt_t)

    print(f"Plotting: {len(ic_bc_pts)} IC/BC points, {len(pred_pts)} predicted points")

    # ---- IC + BC points ----
    if len(ic_bc_pts) > 0:
        ic_bc_arr = np.array(ic_bc_pts)
        ax.scatter(
            ic_bc_arr[:,1], ic_bc_arr[:,0],
            s=20, c='gray', alpha=0.5,
            label='IC/BC', zorder=1
        )

    # ---- Predicted points ----
    if len(pred_pts) > 0:
        pred_pts_arr = np.array(pred_pts)
        pred_vals_arr = np.array(pred_vals)

        scatter = ax.scatter(
            pred_pts_arr[:,1], pred_pts_arr[:,0],
            c=pred_vals_arr, s=40,
            cmap='viridis', alpha=0.8,
            edgecolors='black', linewidth=0.5,
            label='Predicted', zorder=2
        )
        plt.colorbar(scatter, ax=ax, label='Predicted u value')
    else:
        ax.text(
            0.5, 0.5,
            'No predicted interior points to display',
            transform=ax.transAxes,
            ha='center', va='center', fontsize=14
        )

    ax.set_xlabel('x (space)')
    ax.set_ylabel('t (time)')
    ax.set_title('Rollout Results — Predicted Interior Points')
    ax.set_aspect('auto')

    if len(pred_pts) > 0 or len(ic_bc_pts) > 0:
        ax.legend()

    plt.tight_layout()
    display(fig)
    print(f"\n✓ Displayed {len(pred_pts)} predicted points and {len(ic_bc_pts)} IC/BC points")
    return fig, ax

In [ ]:
import pickle

_ROLLOUT_CACHE = "rollout_results_wave.pkl"

def save_rollout_results(
    path=None,
    visited_mlp=None,
    u_mlp=None,
    hist_mlp=None,
    visited_lin=None,
    u_lin=None,
    hist_lin=None,
    data=None,
    cfg_mlp=None,
    metrics_mlp=None,
    metrics_lin=None,
):
    """Save rollout results so you can load later and run visualization without re-running rollouts."""
    path = path or _ROLLOUT_CACHE
    payload = {
        "visited_mlp": visited_mlp,
        "u_mlp": u_mlp,
        "hist_mlp": hist_mlp,
        "visited_lin": visited_lin,
        "u_lin": u_lin,
        "hist_lin": hist_lin,
        "data": data,
        "cfg_mlp": cfg_mlp,
        "metrics_mlp": metrics_mlp,
        "metrics_lin": metrics_lin,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f)
    print(f"Saved rollout results to {path}")

def load_rollout_results(path=None):
    """Load previously saved rollout results. Returns dict with keys:
    visited_mlp, u_mlp, hist_mlp, visited_lin, u_lin, hist_lin, data, cfg_mlp, metrics_mlp, metrics_lin.
    """
    path = path or _ROLLOUT_CACHE
    with open(path, "rb") as f:
        out = pickle.load(f)
    print(f"Loaded rollout results from {path}")
    return out